<a href="https://colab.research.google.com/github/karkhutmaria/KarkhutM_NLP_hw/blob/main/%D0%9A%D0%B0%D1%80%D1%85%D1%83%D1%82_%D0%9C_%2C_fine_tuning_hw.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Домашнее задание

**Датасет:** [`ag_news`](https://huggingface.co/datasets/fancyzhx/ag_news) — классификация новостей по 4-м категориям (World, Sports, Business, Sci/Tech)

**Техническое задание:**

1.  Загрузите датасет `ag_news`
2.  Выберите модель для дообучения (например, `distilbert-base-uncased` или `bert-base-uncased`), `num_labels=4`
3.  Токенизируйте данные (`max_length=128`)
4.  Настройте `TrainingArguments`:
    *   `learning_rate = 2e-5`
    *   `per_device_train_batch_size = 16`
    *   `num_train_epochs = 3`
    *   `eval_strategy = "epoch"`
    *   `save_strategy = "epoch"`
    *   `load_best_model_at_end = True`
    *   `metric_for_best_model = "accuracy"`
5.  Обучите модель с помощью `Trainer`. Для метрик используйте `evaluate.load("accuracy")`
6.  Выведите accuracy на тестовой выборке
7.  Сохраните модель в папку `./ag_news_model`
8.  Протестируйте модель на трех новых новостях (вписать вручную), используя `pipeline`. Выведите предсказанный класс и уверенность модели

In [25]:
!pip install transformers datasets evaluate accelerate gradio -q
!pip install huggingface_hub -q

import torch
print(f"GPU доступен: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Тип GPU: {torch.cuda.get_device_name(0)}")

GPU доступен: True
Тип GPU: Tesla T4


In [26]:
import numpy as np
import torch
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
import evaluate

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [27]:
from datasets import load_dataset

dataset = load_dataset("fancyzhx/ag_news")
print(f"Датасет загружен. Train: {len(dataset['train'])}, Test: {len(dataset['test'])}")

Датасет загружен. Train: 120000, Test: 7600


In [28]:
model_load = "distilbert/distilroberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_load)
model = AutoModelForSequenceClassification.from_pretrained(
    model_load,
    num_labels=4
).to(device)

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: distilbert/distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [29]:
def tokenize(examples):
  return tokenizer(examples["text"], truncation=True, max_length=128)

tokenized_dataset = dataset.map(tokenize, batched=True)
train_dataset = tokenized_dataset["train"]
evaluation_dataset = tokenized_dataset["test"].shuffle(seed=42).select(range(3000))

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [30]:
training_args = TrainingArguments(
    output_dir="./results-ag_news",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="tensorboard",
    logging_steps=500,
)

In [31]:
accuracy = evaluate.load("accuracy")

def metrics(eval_pred):
  predictions, labels = eval_pred
  predictions = np.argmax(predictions, axis=1)
  return accuracy.compute(predictions=predictions, references=labels)

In [32]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=evaluation_dataset,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.209075,0.193117,0.939333
2,0.161128,0.198582,0.942667
3,0.109758,0.210260,0.946000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=22500, training_loss=0.1722645509507921, metrics={'train_runtime': 3096.4189, 'train_samples_per_second': 116.263, 'train_steps_per_second': 7.266, 'total_flos': 8581506895549056.0, 'train_loss': 0.1722645509507921, 'epoch': 3.0})

In [33]:
eval_results = trainer.evaluate()
print(f"\nEvaluation results: {eval_results}")

Training Loss,Validation Loss,Epoch,Accuracy
0.109758,0.210260,3,0.946000



Evaluation results: {'eval_loss': 0.21025972068309784, 'eval_accuracy': 0.946}


In [34]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [35]:
torch.save(model, '/content/drive/MyDrive/ag_news_model.pth')

In [36]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

test_texts = [
    "Jamaica’s beach access crisis: We shouldn’t be forced to fight for what is already ours",
    "From captain to coach: Rod Brind’Amour’s two Stanley Cups with the Hurricanes, 20 years apart",
    "Pokémon Go data trained AI that could assist military drones in war zones",
    "Investment fraud in UK soared to more than £220m lost last year, trade body says"
]

for text in test_texts:
    result = classifier(text)[0]
    print(f"Text: {text}\nSentiment: {result['label']}, Score: {result['score']:.4f}\n")

Text: Jamaica’s beach access crisis: We shouldn’t be forced to fight for what is already ours
Sentiment: LABEL_0, Score: 0.9513

Text: From captain to coach: Rod Brind’Amour’s two Stanley Cups with the Hurricanes, 20 years apart
Sentiment: LABEL_0, Score: 0.6864

Text: Pokémon Go data trained AI that could assist military drones in war zones
Sentiment: LABEL_3, Score: 0.9934

Text: Investment fraud in UK soared to more than £220m lost last year, trade body says
Sentiment: LABEL_2, Score: 0.8936



Тематика всех текстов определена верно, кроме второго текста (должен быть LABEL_1, т.к. тематика новости - спорт), именно поэтому степень уверенности модели во втором тексте намного меньше, чем во всех остальных.